# Data Collection

This notebook fetches macroeconomic and financial data from **Yahoo Finance** and **FRED** (Federal Reserve Economic Data) starting from **January 2000** to **December 2025**. The extended date range covers key economic events such as the dot-com bubble, the 2008 financial crisis, and the COVID-19 pandemic.

| Variable | Source | Series Code |
|---|---|---|
| S&P 500 Index | Yahoo Finance | ^GSPC |
| S&P 500 Energy | Yahoo Finance | XLE |
| S&P 500 IT | Yahoo Finance | XLK |
| S&P 500 Financials | Yahoo Finance | XLF |
| S&P 500 Healthcare | Yahoo Finance | XLV |
| S&P 500 Industrials | Yahoo Finance | XLI |
| CPI | FRED | CPIAUCSL |
| Fed Funds Rate | FRED | FEDFUNDS |
| Unemployment Rate | FRED | UNRATE |
| GDP Growth | FRED | A191RL1Q225SBEA |
| Exchange Rate (USD Index) | Yahoo Finance | DX-Y.NYB |



## 1. Install & Import Libraries

In [1]:
# Install required packages (run once)
!pip install yfinance pandas pandas-datareader fredapi python-dotenv -q


In [2]:
import yfinance as yf                  # For fetching S&P 500 data from Yahoo Finance
import pandas_datareader.data as web    # For fetching data from FRED
from datetime import datetime           # For handling dates
import pandas as pd                     # For data manipulation
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

## 2. Define Date Range

We collect data from **January 2000** up to **December 2025** to ensure we have enough historical data for analysis. You can adjust `START` and `END` as needed.

In [3]:
# Set the date range for all data fetches
START = "2000-01-01"
END = "2025-12-31"

print(f"Date range: {START} to {END}")

Date range: 2000-01-01 to 2025-12-31


## 3. Fetch S&P 500 Sector Indices (Yahoo Finance)

We fetch the **Close** prices for 6 key S&P 500 sector indices from Yahoo Finance:

| Sector | Ticker | Description |
|---|---|---|
| S&P 500 | `^GSPE` | S&P 500 Sector Index |
| Energy | `XLE` | S&P 500 Energy Sector Index |
| Information Technology | `XLK` | S&P 500 IT Sector Index |
| Financials | `XLF` | S&P 500 Financials Sector Index |
| Healthcare | `XLV` | S&P 500 Healthcare Sector Index |
| Industrials | `XLI` | S&P 500 Industrials Sector Index |


> 

In [4]:
# Define S&P 500 sector indices tickers and column names
indices = {
    "SP500": "^GSPC",            
    "SP500_Energy": "XLE",     
    "SP500_IT": "XLK",    
    "SP500_Financials": "XLF",  
    "SP500_Healthcare": "XLV",   
    "SP500_Industrials": "XLI"
}

# Download all sector indices from Yahoo Finance

all_data = []

for name, ticker in indices.items():
    
    print(f"Downloading data for {name} ({ticker})...")
    
    # Download data with a monthly interval
    data = yf.download(
        ticker,
        start=START,
        end=END,
        interval="1mo",   # Monthly interval
        auto_adjust=False,
        progress=False
    )
    
    # Drop rows with all NaN values (if any)
    # data.dropna(how="all", inplace=True)
    
    # Save to CSV file
    file_name = f"{name}_monthly_data.csv"

    # Remove multi-index if exists
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
        
    data.to_csv(f"../data/SP500/{file_name}",index=True)
    
    print(f"Saved: {file_name}")

print("\nAll indices downloaded successfully!")

Saved: SP500_monthly_data.csv
Saved: SP500_Energy_monthly_data.csv
Saved: SP500_IT_monthly_data.csv
Saved: SP500_Financials_monthly_data.csv
Saved: SP500_Healthcare_monthly_data.csv
Saved: SP500_Industrials_monthly_data.csv

All indices downloaded successfully!


In [5]:
from src.data_collection import merge_monthly_csv_files

# Merge data
merged_df = merge_monthly_csv_files("../data/SP500")

merged_df.head()

,Date,SP500_Energy_Open,SP500_Energy_Close,SP500_Energy_Volume,SP500_Healthcare_Open,SP500_Healthcare_Close,SP500_Healthcare_Volume,SP500_Industrials_Open,SP500_Industrials_Close,SP500_Industrials_Volume,SP500_Open,SP500_Close,SP500_Volume,SP500_Financials_Open,SP500_Financials_Close,SP500_Financials_Volume,SP500_IT_Open,SP500_IT_Close,SP500_IT_Volume
0,2000-01-01,13.656250,13.656250,11807200,31.000000,30.109375,1869900,29.484375,27.187500,1609100,1469.250000,1394.459961,21494400000,19.267872,18.734768,10665261,27.812500,25.281250,52014000
1,2000-02-01,13.656250,13.078125,8426000,30.218750,28.140625,1110800,27.156250,25.687500,1350100,1394.459961,1366.420044,20912000000,18.836311,16.729284,10371791,25.257812,27.937500,28690000
2,2000-03-01,13.007813,14.656250,17215200,28.265625,30.671875,2289900,25.703125,29.203125,4562600,1366.420044,1498.579956,26156200000,16.754671,19.712124,18609520,28.000000,30.281250,50668200
3,2000-04-01,14.750000,14.437500,11637800,30.921875,30.312500,253600,29.312500,29.625000,2131100,1498.579956,1452.430054,20106460000,20.130991,19.902517,10647535,29.906250,27.500000,38139400
4,2000-05-01,14.593750,16.132812,10297600,30.406250,29.500000,511400,29.734375,29.500000,560900,1452.430054,1420.599976,19898300000,18.988626,20.346771,6637187,27.906250,24.640625,20860400


In [6]:
# Save processed dataset
merged_df.to_csv("../data/processed/SP500_Master_Monthly.csv", index=False)

## 4. Fetch FRED Data

We fetch 5 macroeconomic variables from FRED using `pandas_datareader` (from 2000 to 2025):

- **CPIAUCSL** — Consumer Price Index for All Urban Consumers (monthly)
- **FEDFUNDS** — Effective Federal Funds Rate (monthly)
- **UNRATE** — Unemployment Rate (monthly)
- **A191RL1Q225SBEA** — Real GDP Growth Rate (quarterly, annualised)
- **USD Index** — Combined from two FRED series

In [7]:
# Define FRED series codes and their readable names (excluding USD Index — handled separately)
fred_series = {
    "CPI": "CPIAUCSL",
    "Interest_Rate": "FEDFUNDS",
    "GDP": "GDP",
    "Unemployment": "UNRATE",
    "VIX": "VIXCLS",
    "BAA": "BAA",
    "AAA": "AAA"
}

In [8]:
# Fetch each FRED series and store in a list
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env and loads vars into environment

api_key = os.getenv("API_KEY")

from fredapi import Fred

data = pd.DataFrame()

fred = Fred(api_key=api_key)

for name, series in fred_series.items():
    temp = fred.get_series(
        series,
        observation_start=START,
        observation_end=END
    )
    
    temp.index = pd.to_datetime(temp.index)

    if series in ["VIXCLS", "BAA", "AAA"]:
        temp = temp.resample("MS").mean()

    data[name] = temp

data["Credit_Spread"] = data["BAA"] - data["AAA"]

data.to_csv("../data/macro_data_fred.csv")


print("Data saved successfully.")

Data saved successfully.


In [9]:
data.head(10)

,CPI,Interest_Rate,GDP,Unemployment,VIX,BAA,AAA,Credit_Spread
2000-01-01,169.3,5.45,10002.179,4.0,23.202000,8.33,7.78,0.55
2000-02-01,170.0,5.73,NaN,4.1,23.595500,8.29,7.68,0.61
2000-03-01,171.0,5.85,NaN,4.0,22.718261,8.37,7.68,0.69
2000-04-01,170.9,6.02,10247.720,3.8,27.164211,8.40,7.64,0.76
2000-05-01,171.2,6.27,NaN,4.0,26.373182,8.90,7.99,0.91
2000-06-01,172.2,6.53,NaN,4.0,21.540000,8.48,7.67,0.81
2000-07-01,172.7,6.54,10318.165,4.0,19.893000,8.35,7.65,0.70
2000-08-01,172.7,6.50,NaN,4.1,18.088696,8.26,7.55,0.71
2000-09-01,173.6,6.52,NaN,3.9,19.687500,8.35,7.62,0.73
2000-10-01,173.9,6.51,10435.744,3.9,25.200000,8.34,7.55,0.79


### 4.1 Fetch USD Index

The Broad USD Index (`DTWEXBGS`) only starts from **2006**. To get coverage from **2000**, we use USD Index (`DX-Y.NYB`, available 1973).

In [10]:

usd = yf.download("DX-Y.NYB", start=START, end=END, auto_adjust=False)

usd.index = pd.to_datetime(usd.index)

# Force Close column to Series safely
usd_close = usd.loc[:, "Close"]

# If it is still DataFrame, squeeze it
usd_close = usd_close.squeeze()

# Now resample
usd_monthly = usd_close.resample("MS").mean()

# Convert to DataFrame only if needed
if isinstance(usd_monthly, pd.Series):
    usd_monthly = usd_monthly.to_frame(name="USD_Index")
else:
    usd_monthly.columns = ["USD_Index"]

usd_monthly.to_csv("../data/usd_index_monthly.csv")

print("USD Index downloaded successfully.")

[*********************100%***********************]  1 of 1 completed

USD Index downloaded successfully.


## 5. Merge All Data

The datasets have different frequencies (daily, monthly, quarterly). We resample everything to **monthly frequency** (end-of-month) so they can be merged into a single DataFrame spanning from **2000 to 2025**.

For the merge, we use macro_data_fred.csv, usd_index_monthly.csv,SP500_Master_Monthly.csv files

In [11]:

# Load CSV files
macro_data = pd.read_csv("../data/macro_data_fred.csv", parse_dates=True, index_col=0)
usd_data = pd.read_csv("../data/usd_index_monthly.csv", parse_dates=True, index_col='Date')
sp500_data = pd.read_csv("../data/processed/SP500_Master_Monthly.csv", parse_dates=True, index_col='Date')

df = macro_data.join(sp500_data, how='outer').join(usd_data, how='outer')
df.index.name = 'Date'
df.sort_index(inplace=True)

df.to_csv("../data/processed/main_data.csv")

## 6. Quick Data Overview

In [12]:
# Display basic statistics
df.describe()



,CPI,Interest_Rate,GDP,Unemployment,VIX,BAA,AAA,Credit_Spread,SP500_Energy_Open,SP500_Energy_Close,...,SP500_Open,SP500_Close,SP500_Volume,SP500_Financials_Open,SP500_Financials_Close,SP500_Financials_Volume,SP500_IT_Open,SP500_IT_Close,SP500_IT_Volume,USD_Index
count,311.000000,312.000000,104.000000,311.000000,312.000000,312.000000,312.000000,312.000000,312.000000,312.000000,...,312.000000,312.000000,3.120000e+02,312.000000,312.000000,3.120000e+02,312.000000,312.000000,3.120000e+02,312.000000
mean,233.174939,2.007788,17898.332769,5.644373,19.837377,5.794006,4.790096,1.003910,30.344765,30.257355,...,2260.297107,2276.552624,7.164216e+10,24.118072,24.211314,1.201167e+09,32.795784,32.787434,2.923547e+08,92.870458
std,41.174192,2.031844,5777.974624,1.944560,7.962719,1.334869,1.250124,0.396935,11.140713,10.608354,...,1445.296627,1467.688303,2.925686e+10,9.798597,9.955487,1.242403e+09,33.343036,32.599331,2.182689e+08,11.270004
min,169.300000,0.050000,10002.179000,3.400000,10.125455,3.160000,2.140000,0.550000,10.600000,10.580000,...,729.570007,735.090027,1.908910e+10,5.938262,6.173842,5.252681e+06,5.970000,5.915000,8.395600e+06,72.116819
25%,201.850000,0.150000,13840.997000,4.200000,14.207024,4.790000,3.870000,0.767500,21.835000,22.150000,...,1217.320007,1217.147522,5.031266e+10,18.070675,18.048334,2.236554e+08,11.090000,11.052500,7.451535e+07,82.824659
50%,231.638000,1.260000,16534.304000,5.000000,17.796220,5.780000,4.955000,0.910000,32.052501,31.872499,...,1522.485046,1528.684998,7.623644e+10,23.155971,23.133261,9.776292e+08,17.590000,17.392500,2.858150e+08,93.397182
75%,255.264500,3.977500,21183.184500,6.250000,23.186250,6.582500,5.510000,1.130000,38.276251,38.401249,...,2899.972473,2917.099976,8.735909e+10,28.154999,28.155108,1.479133e+09,38.954999,39.457500,4.150423e+08,100.235149
max,326.031000,6.540000,31490.070000,14.800000,62.668947,9.210000,7.990000,3.380000,90.940002,50.049999,...,6882.319824,6849.089844,1.621854e+11,53.700001,54.770000,7.904224e+09,217.440002,150.339996,1.564823e+09,118.966501


In [13]:
df.columns

Index(['CPI', 'Interest_Rate', 'GDP', 'Unemployment', 'VIX', 'BAA', 'AAA',
       'Credit_Spread', 'SP500_Energy_Open', 'SP500_Energy_Close',
       'SP500_Energy_Volume', 'SP500_Healthcare_Open',
       'SP500_Healthcare_Close', 'SP500_Healthcare_Volume',
       'SP500_Industrials_Open', 'SP500_Industrials_Close',
       'SP500_Industrials_Volume', 'SP500_Open', 'SP500_Close', 'SP500_Volume',
       'SP500_Financials_Open', 'SP500_Financials_Close',
       'SP500_Financials_Volume', 'SP500_IT_Open', 'SP500_IT_Close',
       'SP500_IT_Volume', 'USD_Index'],
      dtype='object')

In [14]:
# Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")


Missing values per column:
CPI                           1
Interest_Rate                 0
GDP                         208
Unemployment                  1
VIX                           0
BAA                           0
AAA                           0
Credit_Spread                 0
SP500_Energy_Open             0
SP500_Energy_Close            0
SP500_Energy_Volume           0
SP500_Healthcare_Open         0
SP500_Healthcare_Close        0
SP500_Healthcare_Volume       0
SP500_Industrials_Open        0
SP500_Industrials_Close       0
SP500_Industrials_Volume      0
SP500_Open                    0
SP500_Close                   0
SP500_Volume                  0
SP500_Financials_Open         0
SP500_Financials_Close        0
SP500_Financials_Volume       0
SP500_IT_Open                 0
SP500_IT_Close                0
SP500_IT_Volume               0
USD_Index                     0
dtype: int64

Total rows: 312
